In [ ]:
# ============================================
# DEEPFAKE DETECTION - MULTI-DATASET TRAINING
# ============================================
# Training on combined DFDC02 + DFD01 datasets.
# 4 ablation experiments: full, spatial, temporal, sequential.
# Handles DFD01 class imbalance (363 real vs 3068 fake).
#
# PREREQUISITES:
# - Kaggle dataset 1: preprocessed_DFDC02_16 (real/fake face crops)
# - Kaggle dataset 2: preprocessed_DFD01_16 (real/fake face crops)
# Both must be added to the notebook as input datasets.

import subprocess, sys, os

# Step 1: Install dependencies
print("=== Installing dependencies ===")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2", "scipy<1.17", "scikit-learn", "timm", "facenet-pytorch"],
               check=True)
print("Dependencies installed.")

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(["rm", "-rf", "/kaggle/working/project"], check=False)
    subprocess.run(["git", "clone", "https://github.com/Jokeera/deepfake-detection.git",
                     "/kaggle/working/project"], check=True)
    print("Project cloned.")
else:
    subprocess.run(["git", "-C", "/kaggle/working/project", "pull", "--ff-only"],
                   check=True)
    print("Project updated.")

# Step 3: Write multi-dataset training script
train_script = r'''
import os, sys, time, json, gc
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f"numpy version: {np.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU!'}")
assert torch.cuda.is_available(), "No GPU detected!"
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# =============================================
# Find BOTH datasets
# =============================================
print("\n=== Searching for datasets ===")

def find_dataset(keyword):
    """Find preprocessed dataset by keyword in path."""
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'real' in dirs and 'fake' in dirs:
            real_p = os.path.join(root, 'real')
            fake_p = os.path.join(root, 'fake')
            rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
            fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
            if rc > 0 and fc > 0 and keyword.lower() in root.lower():
                return root, rc, fc
    return None, 0, 0

# Try to find both datasets
dfdc02_path, dfdc02_real, dfdc02_fake = find_dataset('dfdc02')
dfd01_path, dfd01_real, dfd01_fake = find_dataset('dfd01')

# Fallback: scan all datasets with real/fake structure
if dfdc02_path is None or dfd01_path is None:
    print("Keyword search failed, scanning all datasets...")
    all_datasets = []
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'real' in dirs and 'fake' in dirs:
            real_p = os.path.join(root, 'real')
            fake_p = os.path.join(root, 'fake')
            rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
            fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
            if rc > 0 and fc > 0:
                all_datasets.append((root, rc, fc))
                print(f"  Found: {root} (real={rc}, fake={fc})")

    if len(all_datasets) < 2:
        print("ERROR: Need at least 2 datasets! Found:")
        for ds in all_datasets:
            print(f"  {ds[0]} (real={ds[1]}, fake={ds[2]})")
        print("\nListing /kaggle/input/:")
        for root, dirs, files in os.walk('/kaggle/input'):
            level = root.replace('/kaggle/input', '').count(os.sep)
            if level < 3:
                indent = '  ' * level
                print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
        sys.exit(1)

    # DFDC02 has more balanced classes, DFD01 is heavily fake-biased
    for path, rc, fc in all_datasets:
        ratio = fc / max(rc, 1)
        if ratio < 3.0 and dfdc02_path is None:
            dfdc02_path, dfdc02_real, dfdc02_fake = path, rc, fc
        elif ratio >= 3.0 and dfd01_path is None:
            dfd01_path, dfd01_real, dfd01_fake = path, rc, fc

if dfdc02_path is None or dfd01_path is None:
    print(f"ERROR: Could not identify both datasets!")
    print(f"  DFDC02: {dfdc02_path}")
    print(f"  DFD01:  {dfd01_path}")
    sys.exit(1)

print(f"\nDFDC02: {dfdc02_path}")
print(f"  Real: {dfdc02_real}, Fake: {dfdc02_fake}, Total: {dfdc02_real + dfdc02_fake}")
print(f"  Ratio fake/real: {dfdc02_fake / max(dfdc02_real, 1):.2f}")

print(f"\nDFD01: {dfd01_path}")
print(f"  Real: {dfd01_real}, Fake: {dfd01_fake}, Total: {dfd01_real + dfd01_fake}")
print(f"  Ratio fake/real: {dfd01_fake / max(dfd01_real, 1):.2f}")

total_real = dfdc02_real + dfd01_real
total_fake = dfdc02_fake + dfd01_fake
print(f"\nCOMBINED: Real={total_real}, Fake={total_fake}, Total={total_real + total_fake}")
print(f"Combined ratio fake/real: {total_fake / max(total_real, 1):.2f}")

# =============================================
# Run experiments with multi-dataset
# =============================================
from config import Config
from train import train
from utils import save_metrics

# Combined dataset path using '+' separator
combined_dataset_root = f"{dfdc02_path}+{dfd01_path}"
combined_dataset_name = "dfdc02+dfd01"

# Unique split filename for multi-dataset
SPLIT_FILENAME = "split_multi_seed42.json"

# Compute pos_weight for class imbalance compensation
pos_weight_value = total_real / max(total_fake, 1)
print(f"\npos_weight (real/fake): {pos_weight_value:.4f}")
if pos_weight_value > 3.0:
    pos_weight_value = 3.0
    print(f"pos_weight capped to {pos_weight_value}")
elif pos_weight_value < 0.33:
    pos_weight_value = 0.33
    print(f"pos_weight capped to {pos_weight_value}")

# Use weighted sampler when class imbalance is significant
use_sampler = abs(total_fake / max(total_real, 1) - 1.0) > 0.5
print(f"Using weighted sampler: {use_sampler}")

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',    'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

all_results = []
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*70}")
    print(f"[{i}/4] {exp['name']} (model_type={exp['model_type']})")
    print(f"{'='*70}\n")

    cfg = Config()
    # Multi-dataset paths
    cfg.dataset_root = combined_dataset_root
    cfg.dataset_name = combined_dataset_name

    # Model config
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']

    # Training config
    cfg.max_epochs = 30
    cfg.batch_size = 16
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.seed = 42
    cfg.unfreeze_last_n_blocks = 2

    # Class imbalance handling
    if use_sampler:
        cfg.use_weighted_sampler = True
        cfg.use_class_weights = False
    else:
        cfg.use_weighted_sampler = False
        cfg.pos_weight = round(pos_weight_value, 4) if abs(pos_weight_value - 1.0) > 0.2 else None

    # Split strategy: first experiment creates split, rest reuse it
    cfg.splits_dir = './splits'
    cfg.split_filename = SPLIT_FILENAME
    cfg.output_dir = './experiments'

    split_file = os.path.join(cfg.splits_dir, SPLIT_FILENAME)
    if i == 1:
        # First experiment: create random split and save it
        cfg.split_mode = 'random'
        cfg.save_split = True
        print(f"Creating new split: {split_file}")
    else:
        # Subsequent experiments: reuse the same split
        cfg.split_mode = 'fixed'
        cfg.save_split = False
        print(f"Reusing existing split: {split_file}")

    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        metrics['training_datasets'] = ['dfdc02', 'dfd01']
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f"\n[OK] {exp['name']} done in {metrics['duration_min']} min")
        print(f"     AUC={test.get('auc',0):.4f}  Acc={test.get('accuracy',0):.4f}  F1={test.get('f1',0):.4f}")
    except Exception as e:
        elapsed = round((time.time() - t0) / 60, 1)
        print(f"\n[FAIL] {exp['name']} after {elapsed} min: {e}")
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': elapsed,
        })

    # Cleanup between experiments
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
          f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

    save_metrics(all_results, './experiments/all_results_multi.json')

# =============================================
# Summary
# =============================================
print(f"\n{'='*70}")
print("MULTI-DATASET TRAINING RESULTS SUMMARY")
print(f"Training on: DFDC02 ({dfdc02_real + dfdc02_fake} videos) + DFD01 ({dfd01_real + dfd01_fake} videos)")
print(f"{'='*70}")
print(f"{'Experiment':<25} {'AUC':>8} {'Acc':>8} {'F1':>8} {'EER':>8} {'Epoch':>6} {'Time':>8}")
print('-' * 75)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f"{r['experiment_name']:<25} {t.get('auc',0):>8.4f} {t.get('accuracy',0):>8.4f} "
              f"{t.get('f1',0):>8.4f} {t.get('eer',0):>8.4f} {be:>6} {r.get('duration_min',0):>7.1f}m")
    else:
        print(f"{r['experiment_name']:<25} {'FAILED':>8} {r.get('error','')[:40]}")

# Copy results for download
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system("cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null")
os.system("cp -r splits/ /kaggle/working/splits/ 2>/dev/null")
os.system("tar czf /kaggle/working/results_multi.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null")
print(f"\nresults_multi.tar.gz ready for download")
'''

with open('/kaggle/working/run_multi_training.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training in FRESH subprocess
print("\n=== Starting multi-dataset training in subprocess ===")
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_multi_training.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f"\nTraining subprocess exited with code: {result.returncode}")

# Copy results
if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print("Results copied to /kaggle/working/experiments/")
if os.path.exists('/kaggle/working/project/splits'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/splits/', '/kaggle/working/splits/'])
    print("Splits copied to /kaggle/working/splits/")
if os.path.exists('/kaggle/working/results_multi.tar.gz'):
    print("results_multi.tar.gz is ready for download")